# Auto-encodeurs débruiteurs et variationnels avec Keras

Le notebook précédent entraînait des auto-encodeurs à reconstruire leur entrée. Celui-ci change d'abord la tâche : retrouver une image propre à partir d'une version bruitée. Puis il change le modèle, pour pouvoir générer de nouvelles images. Les données sont celles de Fashion-MNIST : 60 000 images d'entraînement et 10 000 de test, 28 × 28 en niveaux de gris comme MNIST, mais de 10 sortes de vêtements et d'accessoires.

1. Corrompre les images
2. Un débruiteur dense
3. Un code plus large
4. Un débruiteur convolutif
5. Un auto-encodeur variationnel
6. Générer des images

Chaque partie réutilise les données et les modèles des précédentes. Pas besoin de GPU : le notebook tourne en une vingtaine de minutes sur un processeur, dont près de la moitié pour le débruiteur convolutif de la partie 4. Un GPU (`Exécution > Modifier le type d'exécution`) l'accélère nettement.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers, ops

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = [
  "T-shirt",
  "Pantalon",
  "Pull",
  "Robe",
  "Manteau",
  "Sandale",
  "Chemise",
  "Basket",
  "Sac",
  "Bottine",
]
input_dim = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255
print(X_train.shape, X_test.shape)


def show_images(*rows: np.ndarray, titles: list | np.ndarray | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title, fontsize=8)
  plt.show()


show_images(X_train[:10], titles=[class_names[y] for y in y_train[:10]])

## 1. Corrompre les images

Un débruiteur apprend à retrouver l'image propre à partir d'une version abîmée : ses exemples d'entraînement sont des paires (image bruitée, image propre). Fabriquez les images bruitées :

- `X_train_noisy` et `X_test_noisy` : `X_train` et `X_test`, plus un bruit gaussien de moyenne 0 et d'écart-type 0,5 tiré indépendamment pour chaque pixel ;
- ramenez ensuite chaque pixel dans [0, 1], comme ceux d'une vraie image.

La cellule après la solution affiche des images bruitées au-dessus des images propres.

**Documentation utile**

- [`numpy.random.Generator.normal`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html)
- [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `rng = np.random.default_rng(0)`, puis `rng.normal(0, 0.5, size=X_train.shape)` : un bruit de la forme des images, un tirage par pixel
- `np.clip(X, 0, 1)` : les valeurs sous 0 deviennent 0, celles au-dessus de 1 deviennent 1
- vérification : `X_train_noisy.min()` vaut 0, `X_train_noisy.max()` vaut 1, et `X_train_noisy.shape` est celle de `X_train`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rng = np.random.default_rng(0)
X_train_noisy = np.clip(X_train + rng.normal(...), ..., ...)
X_test_noisy = ...
```

### Solution

In [ ]:
rng = np.random.default_rng(0)
X_train_noisy = np.clip(X_train + rng.normal(0, 0.5, size=X_train.shape), 0, 1)
X_test_noisy = np.clip(X_test + rng.normal(0, 0.5, size=X_test.shape), 0, 1)

### Les images bruitées

In [ ]:
titles = [class_names[y] for y in y_test[:10]]
show_images(X_test_noisy[:10], X_test[:10], titles=titles)

## 2. Un débruiteur dense

Construisez `autoencoder`, un débruiteur dense à une couche cachée :

- une couche `Dense` de `encoding_dim = 12` unités et d'activation ReLU, le code, sur des entrées de taille `input_dim` ;
- la couche de sortie, de `input_dim` unités, avec une activation adaptée à des pixels dans [0, 1] ;
- compilé avec l'optimiseur Adam et la perte `"mse"`.

La cellule après la solution l'entraîne sur les paires (image bruitée, image propre), puis affiche des images de test bruitées, leur version débruitée et l'image propre.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un seul `keras.Sequential` suffit : `keras.Input(shape=(input_dim,))`, puis les deux couches `Dense`
- la sortie : `activation="sigmoid"`, qui rend des valeurs dans ]0, 1[, comme les pixels
- `autoencoder.summary()` : 9 420 paramètres pour la couche du code (784 × 12 + 12) et 10 192 pour la sortie

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 12
autoencoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(...), layers.Dense(...)],
  name="autoencoder",
)
autoencoder.compile(...)
autoencoder.summary()
```

### Solution

In [ ]:
encoding_dim = 12
autoencoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(encoding_dim, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="autoencoder",
)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

### Entraînement

L'entrée est l'image bruitée, la cible l'image propre. En haut, les images bruitées, puis leur version débruitée, puis les images propres.

In [ ]:
autoencoder.fit(
  X_train_noisy,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test_noisy, X_test),
  verbose=2,
)
X_test_denoised = autoencoder.predict(X_test_noisy, verbose=0)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

## 3. Un code plus large

Un code de 12 nombres limite ce que le débruiteur peut restituer. Refaites-le avec un code de `encoding_dim = 64` unités, entraînez-le comme dans la partie 2 (le même `fit`), et affichez les mêmes images.

La perte de validation baisse-t-elle ? Les images sont-elles plus nettes ? Le réseau recopie-t-il le bruit, maintenant qu'il aurait la place de le faire ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- la solution de la partie 2 avec `encoding_dim = 64`, puis la cellule d'entraînement de la partie 2, copiées l'une après l'autre
- comparez la dernière `val_loss` à celle de la partie 2

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 64
autoencoder = ...
autoencoder.compile(...)
autoencoder.fit(...)
X_test_denoised = ...
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)
```

### Solution

In [ ]:
encoding_dim = 64
autoencoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(encoding_dim, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="autoencoder",
)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.fit(
  X_train_noisy,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test_noisy, X_test),
  verbose=2,
)
X_test_denoised = autoencoder.predict(X_test_noisy, verbose=0)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

La perte de validation baisse nettement, d'environ 0,026 à 0,018, et les images sont plus nettes : on distingue les cols, les manches et les motifs. Le réseau ne recopie pas le bruit, alors qu'il aurait maintenant la place de le faire : sa cible est l'image propre, et chaque pixel de bruit recopié augmenterait la perte. C'est ce qui permet à un débruiteur d'avoir un code large sans apprendre l'identité, contrairement à un auto-encodeur ordinaire.

## 4. Un débruiteur convolutif

Une couche dense traite chaque pixel à part ; une convolution exploite le voisinage de chaque pixel, ce qui convient mieux à une image. Les couches de convolution attendent une image 28 × 28 × 1, avec un axe pour les canaux (un seul en niveaux de gris, trois en couleur) : la cellule suivante remet les images dans cette forme.

In [ ]:
X_train_img = X_train.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)
X_train_noisy_img = X_train_noisy.reshape(-1, 28, 28, 1)
X_test_noisy_img = X_test_noisy.reshape(-1, 28, 28, 1)
print(X_train_img.shape)

### Le modèle

Construisez un `encoder` et un `decoder` entièrement convolutifs, puis `autoencoder`, qui les enchaîne. Toutes les `Conv2D` ont des filtres 3 × 3 et `padding="same"` :

- `encoder`, sur des entrées 28 × 28 × 1 : une `Conv2D` de 16 filtres (ReLU) et un `MaxPooling2D` 2 × 2, puis une `Conv2D` de 8 filtres (ReLU) et un `MaxPooling2D` 2 × 2. Le code est le tableau 7 × 7 × 8 qui en sort ;
- `decoder`, le chemin inverse : une `Conv2D` de 8 filtres (ReLU) et un `UpSampling2D` 2 × 2, une `Conv2D` de 16 filtres (ReLU) et un `UpSampling2D` 2 × 2, puis une dernière `Conv2D` à 1 filtre et activation sigmoïde, qui rend une image 28 × 28 × 1 ;
- `autoencoder`, compilé avec l'optimiseur Adam et la perte `"binary_crossentropy"`.

Vérifiez les formes avec `autoencoder.summary(expand_nested=True)`, et comparez son nombre de paramètres à celui du débruiteur dense de la partie 3. Pourquoi ne pas finir le décodeur par l'`UpSampling2D` ?

**Documentation utile**

- [`keras.layers.UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Conv2D(16, 3, activation="relu", padding="same")` : `padding="same"` garde la taille de l'image ; une petite fonction `conv(filters, activation="relu")` qui la renvoie évite de répéter ces arguments
- `layers.MaxPooling2D(2)` divise la hauteur et la largeur par 2 (28 → 14 → 7), `layers.UpSampling2D(2)` les multiplie par 2 en répétant chaque pixel (7 → 14 → 28)
- le décodeur prend des entrées `keras.Input(shape=(7, 7, 8))`, la forme du code
- la dernière `Conv2D` : `conv(1, "sigmoid")` ; un `UpSampling2D` en fin de réseau rendrait des blocs de 2 × 2 pixels identiques, et une sortie sans sigmoïde pourrait sortir de [0, 1]
- `summary` : la sortie de l'encodeur est `(None, 7, 7, 8)`, celle du décodeur `(None, 28, 28, 1)` ; 3 217 paramètres en tout, contre 101 200 pour le débruiteur dense de la partie 3 : chaque filtre 3 × 3 sert en tout point de l'image

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conv(filters: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filters, 3, activation=activation, padding="same")


encoder = keras.Sequential(
  [keras.Input(shape=(28, 28, 1)), conv(16), layers.MaxPooling2D(2), ..., ...],
  name="encoder",
)
decoder = keras.Sequential(
  [keras.Input(shape=(7, 7, 8)), conv(8), layers.UpSampling2D(2), ..., ..., ...],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(...)
autoencoder.summary(expand_nested=True)
```

#### Solution

In [ ]:
def conv(filters: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filters, 3, activation=activation, padding="same")


encoder = keras.Sequential(
  [
    keras.Input(shape=(28, 28, 1)),
    conv(16),
    layers.MaxPooling2D(2),  # 14 × 14 × 16
    conv(8),
    layers.MaxPooling2D(2),  # 7 × 7 × 8, le code
  ],
  name="encoder",
)
decoder = keras.Sequential(
  [
    keras.Input(shape=(7, 7, 8)),
    conv(8),
    layers.UpSampling2D(2),  # 14 × 14 × 8
    conv(16),
    layers.UpSampling2D(2),  # 28 × 28 × 16
    # Lisse les blocs de l'UpSampling2D, sortie dans ]0, 1[
    conv(1, "sigmoid"),
  ],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary(expand_nested=True)

### Entraînement

In [ ]:
autoencoder.fit(
  X_train_noisy_img,
  X_train_img,
  epochs=5,
  batch_size=256,
  validation_data=(X_test_noisy_img, X_test_img),
  verbose=2,
)
X_test_denoised = autoencoder.predict(X_test_noisy_img, verbose=0)
# L'erreur quadratique moyenne, la perte des parties 2 et 3
print(f"MSE : {np.mean((X_test_denoised - X_test_img) ** 2):.4f}")
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

### Sur des images non bruitées

Le débruiteur n'a vu que des entrées bruitées. Que rend-il d'une image propre ?

In [ ]:
X_test_rebuilt = autoencoder.predict(X_test_img[:10], verbose=0)
show_images(X_test[:10], X_test_rebuilt, titles=titles)

Sur une image propre, le débruiteur rend la forme du vêtement, mais lissée : il efface les détails fins, comme les logos, les carreaux ou les rayures, et force le contraste. Il a appris à garder ce qui ressemble à un vêtement et à effacer le reste ; un détail fin, à ses yeux, ressemble à du bruit. Son erreur quadratique moyenne sur les images bruitées, environ 0,021, se place entre celles des deux débruiteurs denses (0,026 et 0,018), avec 3 217 paramètres au lieu de 19 612 et 101 200 : chaque filtre 3 × 3 sert en tout point de l'image, quand chaque poids d'une couche dense ne relie qu'un pixel à une unité.

## 5. Un auto-encodeur variationnel

Le notebook précédent l'a montré : décoder un code tiré au hasard ne donne pas une image vraisemblable, car les codes d'un auto-encodeur n'occupent qu'une petite partie de l'espace latent, sans forme connue. Un auto-encodeur variationnel (VAE) encode chaque image en une loi, $\mathcal{N}(\mu, \sigma^2)$, et une perte KL pousse ces lois vers $\mathcal{N}(0, I)$ : on peut alors tirer un code dans $\mathcal{N}(0, I)$ et le décoder.

La couche `Sampling` de la slide « Couche d'échantillonnage en Keras » est fournie. Elle reçoit `z_mean` ($\mu$) et `z_log_var` ($\log \sigma^2$), tire $z = \mu + \sigma \varepsilon$ avec $\varepsilon \sim \mathcal{N}(0, I)$, et ajoute la KL à la perte du modèle, avec un poids `beta`.

In [ ]:
class Sampling(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # poids de la KL
    self.seed_generator = keras.random.SeedGenerator(0)

  def call(self, inputs):
    z_mean, z_log_var = inputs
    kl = -0.5 * ops.sum(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var), axis=1)
    self.add_loss(self.beta * ops.mean(kl))  # + perte du modèle
    eps = keras.random.normal(ops.shape(z_mean), seed=self.seed_generator)
    return z_mean + ops.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε


# beta = 1 / 784 : la perte binary_crossentropy est une moyenne sur les 784 pixels,
# alors que la KL est une somme sur les dimensions du code

### Le modèle

Assemblez le VAE de la slide « Un VAE en Keras », avec un espace latent à `latent_dim = 2` dimensions :

- l'encodeur, en API fonctionnelle : une entrée de taille `input_dim`, une `Dense` de 256 unités (ReLU), puis deux `Dense` de `latent_dim` unités sans activation, `z_mean` et `z_log_var`, et `z`, la sortie de `Sampling` sur les deux ; `encoder` est le `keras.Model` qui rend `[z_mean, z_log_var, z]` ;
- `decoder`, un `keras.Sequential` : une `Dense` de 256 unités (ReLU), puis la sortie de `input_dim` unités en sigmoïde ;
- `vae`, le `keras.Model` de l'entrée à `decoder(z)`, compilé avec Adam et la perte `"binary_crossentropy"`, à laquelle `Sampling` ajoute la KL.

La cellule après la solution l'entraîne sur les images propres, puis place les images de test dans l'espace latent.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- en API fonctionnelle, chaque couche s'applique au tenseur précédent : `inputs = keras.Input(shape=(input_dim,))`, puis `h = layers.Dense(256, activation="relu")(inputs)`
- `z_mean` et `z_log_var` s'appliquent tous deux à `h` ; `z = Sampling()([z_mean, z_log_var])`
- `keras.Model(inputs, [z_mean, z_log_var, z], name="encoder")` : un modèle à une entrée et trois sorties
- `vae = keras.Model(inputs, decoder(z), name="vae")` : appeler `decoder` sur `z` le branche à la suite de l'encodeur
- `vae.summary()` : 404 244 paramètres en tout

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
latent_dim = 2
inputs = keras.Input(shape=(input_dim,))
h = layers.Dense(256, activation="relu")(inputs)
z_mean = layers.Dense(latent_dim)(h)
z_log_var = ...
z = Sampling()(...)
encoder = keras.Model(inputs, [...], name="encoder")
decoder = keras.Sequential([keras.Input(shape=(latent_dim,)), ..., ...], name="decoder")
vae = keras.Model(inputs, ..., name="vae")
vae.compile(...)
vae.summary()
```

#### Solution

In [ ]:
latent_dim = 2
inputs = keras.Input(shape=(input_dim,))
h = layers.Dense(256, activation="relu")(inputs)
z_mean = layers.Dense(latent_dim)(h)
z_log_var = layers.Dense(latent_dim)(h)  # log σ², un réel
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(inputs, [z_mean, z_log_var, z], name="encoder")
decoder = keras.Sequential(
  [
    keras.Input(shape=(latent_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="decoder",
)
vae = keras.Model(inputs, decoder(z), name="vae")
vae.compile(optimizer="adam", loss="binary_crossentropy")  # + KL
vae.summary()

### Entraînement et espace latent

La cible est l'image propre elle-même. Chaque point du nuage est le $\mu$ d'une image de test, coloré selon sa classe.

In [ ]:
vae.fit(
  X_train,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
test_means, _, _ = encoder.predict(X_test, verbose=0)
plt.figure(figsize=(7, 6))
for label, name in enumerate(class_names):
  points = test_means[y_test == label]
  plt.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=name)
plt.legend(markerscale=5, fontsize=8)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")
plt.show()

Les μ des images de test restent presque tous entre -3 et 3 sur chaque axe, autour de l'origine : la KL tire chaque loi vers $\mathcal{N}(0, I)$. Chaque classe occupe sa région. Les chaussures (sandales, baskets, bottines) remplissent le bas du nuage, séparées du reste par une bande presque vide ; les sacs sont au milieu à gauche, les pantalons en haut à droite, à côté des robes et des T-shirts. Les autres hauts (pulls, manteaux, chemises) se chevauchent en haut à gauche : ils se ressemblent, et deux dimensions ne suffisent pas à les séparer.

## 6. Générer des images

Tirez 16 codes dans $\mathcal{N}(0, I)$, décodez-les avec `decoder`, et affichez les images obtenues, sur une rangée.

Ressemblent-elles à des vêtements ? Comparez-les aux images décodées depuis du bruit dans le notebook précédent.

**Documentation utile**

- [`keras.random.normal`](https://keras.io/api/random/random_ops/#normal-function)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.random.normal((16, latent_dim), seed=0)` : 16 codes, chaque composante tirée dans $\mathcal{N}(0, 1)$
- `decoder.predict(z_new, verbose=0)` : 16 images de 784 pixels, que `show_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
z_new = keras.random.normal(...)
generated = ...
show_images(generated)
```

### Solution

In [ ]:
z_new = keras.random.normal((16, latent_dim), seed=0)  # z ~ N(0, I)
generated = decoder.predict(z_new, verbose=0)
show_images(generated)

La plupart ressemblent à des vêtements : des hauts, des pantalons, des chaussures, flous comme toutes les images d'un VAE à deux dimensions. Quelques-unes, pâles, mélangent un pantalon et un haut : leur code est tombé entre deux régions, où le décodeur hésite. Dans le notebook précédent, un code tiré au hasard ne donnait que des taches : ici, la KL a placé les codes des images d'entraînement là où $\mathcal{N}(0, I)$ tire les siens, et le décodeur sait les décoder. Un espace latent plus grand donnerait des images plus nettes, mais on ne pourrait plus en dessiner la carte.

### La carte de l'espace latent

Avec 2 dimensions, on peut décoder une grille régulière de codes, de -2 à 2 sur chaque axe, là où $\mathcal{N}(0, I)$ tire presque tous ses points : chaque case montre ce que le décodeur rend en ce point. Comparez avec le nuage de la partie 5.

In [ ]:
n = 12
grid = np.linspace(-2, 2, n)
# z_2 décroît de haut en bas, comme l'axe vertical du nuage
codes = np.array([[z1, z2] for z2 in grid[::-1] for z1 in grid])
images = decoder.predict(codes, verbose=0).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.transpose(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()